In [ ]:
!git clone https://github.com/bremsstrahlung-57/practicum-project

In [ ]:
!pip install torch-pruning wandb --quiet

In [ ]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

import wandb
wandb.login(WANDB_API_KEY)

# Structured Pruning + Quantization — ResNet-18 CIFAR-10
Pipeline: Load baseline → Structured prune → Fine-tune → PTQ INT8 → Benchmark on CPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch_pruning as tp
import time
import tracemalloc
import copy
import os
import wandb
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {DEVICE}')
print(f'Benchmarking will be forced to CPU')

## 1. Model Definition (same CIFAR-10 adapted ResNet-18)

In [ ]:
def get_cifar_resnet18():
    model = torchvision.models.resnet18(weights=None)
    # CIFAR-10 adaptation: replace 7x7 stride-2 conv + maxpool with 3x3 stride-1, no maxpool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)
    return model

model = get_cifar_resnet18()
checkpoint = torch.load('/content/practicum-project/resnet18_cifar10_baseline.pth', map_location='cpu')

# Handle checkpoint formats (state_dict directly or wrapped)
state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))
model.load_state_dict(state_dict)
model.eval()
print('Baseline loaded.')

## 2. Data Loaders

In [ ]:
BATCH_SIZE = 128

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Small calibration set for PTQ (500 samples)
calib_dataset = torch.utils.data.Subset(train_dataset, list(range(500)))
calib_loader  = torch.utils.data.DataLoader(calib_dataset, batch_size=64, shuffle=False)

## 3. Utility Functions

In [ ]:
def evaluate(model, loader, device='cpu', is_quantized=False):
    model.eval()
    if not is_quantized:
        model = model.to(device)
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            if not is_quantized:
                images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return 100 * correct / total


def benchmark_cpu(model, n_runs=100, warmup=20, is_quantized=False):
    """Returns avg latency (ms) and peak RAM (MB) on CPU, single-sample."""
    model = model.cpu().eval() if is_quantized else copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)

    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            model(dummy)

    # Latency
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(dummy)
            times.append((time.perf_counter() - t0) * 1000)

    avg_ms = np.mean(times)

    # RAM
    tracemalloc.start()
    with torch.no_grad():
        model(dummy)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return avg_ms, peak / 1024**2  # ms, MB


def model_size_mb(model):
    """Save to temp file and measure disk size."""
    torch.save(model.state_dict(), '/tmp/_tmp_model.pth')
    size = os.path.getsize('/tmp/_tmp_model.pth') / 1024**2
    os.remove('/tmp/_tmp_model.pth')
    return size


def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6  # millions


def count_macs(model):
    """Rough FLOPs count via torch-pruning."""
    example = torch.randn(1, 3, 32, 32)
    macs, _ = tp.utils.count_ops_and_params(model.cpu(), example)
    return macs / 1e6  # MMACs

## 4. Baseline Metrics (before any pruning)

In [ ]:
wandb.init(
    project='cnn-compression', name='structured-pruning-baseline-ref', group="structured_pruning",

    config={'phase': 'structured_pruning'})

baseline_acc    = evaluate(model, test_loader, device='cpu')
baseline_lat, baseline_ram = benchmark_cpu(model)
baseline_size   = model_size_mb(model)
baseline_params = count_params(model)
baseline_macs   = count_macs(model)

print(f'Accuracy  : {baseline_acc:.2f}%')
print(f'Latency   : {baseline_lat:.2f} ms')
print(f'RAM       : {baseline_ram:.2f} MB')
print(f'Size      : {baseline_size:.2f} MB')
print(f'Params    : {baseline_params:.2f}M')
print(f'MACs      : {baseline_macs:.2f}M')

wandb.log({
    'pruning_ratio': 0.0,
    'accuracy': baseline_acc,
    'latency_ms': baseline_lat,
    'ram_mb': baseline_ram,
    'model_size_mb': baseline_size,
    'params_M': baseline_params,
    'macs_M': baseline_macs,
})
wandb.finish()

## 5. Structured Pruning + Fine-tune Loop
Pruning ratios tested: 30%, 50%, 70% channel removal per layer.

In [ ]:
PRUNING_RATIOS  = [0.3, 0.5, 0.7]   # fraction of channels to remove
FINETUNE_EPOCHS = 15
FINETUNE_LR     = 1e-4

def structured_prune(model, pruning_ratio):
    """
    Remove `pruning_ratio` fraction of channels globally using L1-norm importance.
    torch-pruning handles ResNet skip-connection coupling automatically.
    """
    pruned_model = copy.deepcopy(model).cpu()
    example_input = torch.randn(1, 3, 32, 32)

    # Importance criterion: L1 norm of filter weights (matches unstructured phase framing)
    importance = tp.importance.MagnitudeImportance(p=1)

    pruner = tp.pruner.MagnitudePruner(
        model           = pruned_model,
        example_inputs  = example_input,
        importance      = importance,
        pruning_ratio   = pruning_ratio,
        # Don't prune the classifier head
        ignored_layers  = [pruned_model.fc],
    )

    pruner.step()   # single-shot structured pruning
    return pruned_model


def finetune(model, train_loader, epochs, lr, device):
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            acc = evaluate(model, test_loader, device=str(device))
            print(f'  Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%')

    return model


from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

def apply_ptq_fx(model, calib_loader):
    model = copy.deepcopy(model).cpu().eval()

    torch.backends.quantized.engine = "fbgemm"

    qconfig = get_default_qconfig_mapping("fbgemm").set_global(
    torch.ao.quantization.get_default_qconfig("fbgemm"))
    example_inputs = torch.randn(1, 3, 32, 32)

    prepared = prepare_fx(model, qconfig, example_inputs)

    # calibration
    prepared.eval()
    with torch.no_grad():
        for images, _ in calib_loader:
            prepared(images)

    quantized = convert_fx(prepared)
    return quantized

In [ ]:
results = []

for ratio in PRUNING_RATIOS:
    run_name = f'structured-prune-{int(ratio*100)}pct_fx2'
    print(f'\n{"="*55}')
    print(f'Pruning ratio: {ratio*100:.0f}%  |  Run: {run_name}')
    print('='*55)

    wandb.init(
        project='cnn-compression',
        name=run_name,
        group="structured_pruning",
        config={
        'phase': 'structured_pruning',
        'pruning_ratio': ratio,
        'finetune_epochs': FINETUNE_EPOCHS,
        'finetune_lr': FINETUNE_LR,
    })

    # --- Prune ---
    print('Pruning...')
    pruned = structured_prune(model, ratio)
    acc_before_ft = evaluate(pruned, test_loader, device='cpu')
    print(f'Acc before fine-tune: {acc_before_ft:.2f}%')

    wandb.log({'accuracy_before_finetune': acc_before_ft, 'pruning_ratio': ratio})

    # --- Fine-tune ---
    print(f'Fine-tuning {FINETUNE_EPOCHS} epochs...')
    pruned = finetune(pruned, train_loader, FINETUNE_EPOCHS, FINETUNE_LR, DEVICE)
    acc_after_ft = evaluate(pruned, test_loader, device='cpu')
    print(f'Acc after fine-tune:  {acc_after_ft:.2f}%')

    # --- FP32 Benchmark ---
    lat_fp32, ram_fp32   = benchmark_cpu(pruned)
    size_fp32            = model_size_mb(pruned)
    params_fp32          = count_params(pruned)
    macs_fp32            = count_macs(pruned)

    print(f'[FP32] Latency: {lat_fp32:.2f} ms | Size: {size_fp32:.2f} MB | Params: {params_fp32:.2f}M | MACs: {macs_fp32:.1f}M')

    # Save fine-tuned pruned model (pre-quantization)
    ft_ckpt_path = f'/content/practicum-project/structured_pruned_{int(ratio*100)}pct_fp32.pth'

    torch.save({
        'model_state_dict': pruned.state_dict(),
        'pruning_ratio': ratio,
        'accuracy': acc_after_ft
    }, ft_ckpt_path)

    print(f'FP32 model saved: {ft_ckpt_path}')

    # --- PTQ INT8 ---
    print('Applying PTQ...')
    try:
        q_pruned           = apply_ptq_fx(pruned, calib_loader)
        acc_int8           = evaluate(q_pruned, test_loader, is_quantized=True)
        lat_int8, ram_int8 = benchmark_cpu(q_pruned, is_quantized=True)
        size_int8          = model_size_mb(q_pruned)
        ptq_success        = True
        print(f'[INT8] Acc: {acc_int8:.2f}% | Latency: {lat_int8:.2f} ms | Size: {size_int8:.2f} MB')
    except Exception as e:
        print(f'PTQ failed: {e}')
        acc_int8 = lat_int8 = ram_int8 = size_int8 = None
        ptq_success = False

    # --- Save pruned checkpoint ---
    q_ckpt_path = f'/content/practicum-project/structured_pruned_{int(ratio*100)}pct_int8.pt'

    scripted_model = torch.jit.script(q_pruned)
    torch.jit.save(scripted_model, q_ckpt_path)

    print(f'INT8 model saved: {q_ckpt_path}')

    # --- Log to WandB ---
    log = {
        'pruning_ratio'          : ratio,
        'accuracy_after_finetune': acc_after_ft,
        'latency_fp32_ms'        : lat_fp32,
        'ram_fp32_mb'            : ram_fp32,
        'model_size_fp32_mb'     : size_fp32,
        'params_M'               : params_fp32,
        'macs_M'                 : macs_fp32,
        'speedup_vs_baseline'    : baseline_lat / lat_fp32,
    }
    if ptq_success:
        log.update({
            'accuracy_int8'            : acc_int8,
            'latency_int8_ms'          : lat_int8,
            'ram_int8_mb'              : ram_int8,
            'model_size_int8_mb'       : size_int8,
            'speedup_int8_vs_baseline' : baseline_lat / lat_int8,
        })

    wandb.log(log)
    wandb.finish()

    results.append({'ratio': ratio, **log})

print('\nAll ratios done.')

## 6. Summary Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

ratio_col = 'ratio' if 'ratio' in df.columns else 'pruning_ratio'

df.insert(0, 'model', [f'Structured-{int(r*100)}%' for r in df[ratio_col]])

print('\n=== Structured Pruning + PTQ Results ===')

cols = [
    'model',

    # Accuracy
    'accuracy_after_finetune',
    'accuracy_int8',

    # Latency
    'latency_fp32_ms',
    'latency_int8_ms',

    # Size
    'model_size_fp32_mb',
    'model_size_int8_mb',

    # Structure
    'params_M',
    'macs_M',

    # Speedups
    'speedup_vs_baseline',
    'speedup_int8_vs_baseline'
]

cols = [c for c in cols if c in df.columns]

print(df[cols].to_string(index=False))
print(
    f'\nBaseline (FP32): '
    f'acc={baseline_acc:.2f}%, '
    f'lat={baseline_lat:.2f}ms, '
    f'size={baseline_size:.2f}MB, '
    f'params={baseline_params:.2f}M, '
    f'macs={baseline_macs:.1f}M'
)